# Credit Card Fraud Detection
### Handling Imbalanced Datasets in a Production-Relevant Context

---

## Business Problem

A European card issuer processes hundreds of thousands of transactions daily. A tiny fraction — roughly **0.17%** — are fraudulent. The challenge is not simply detecting fraud; any model that flags every transaction as fraud would catch 100% of cases but be completely unusable in practice.

The real question is:

> **Can we build a model that catches the majority of fraudulent transactions while keeping false positive alerts low enough that the operations team can actually act on them?**

This analysis works through that problem end-to-end: understanding the imbalance, engineering a balanced training set, identifying the most discriminative features, and evaluating models on metrics that actually reflect business reality.

---

## Dataset

- **Source:** [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud)
- **Rows:** 284,807 transactions over two days in September 2013
- **Features:** 30 total — `Time`, `Amount`, and V1–V28 (PCA-transformed, anonymised)
- **Target:** `Class` — 0 = legitimate, 1 = fraud
- **Imbalance:** 492 fraud cases out of 284,807 total (0.17%)

**Note on V1–V28:** These features have been transformed using PCA for confidentiality. We cannot interpret them in business terms, but we can identify which ones best separate fraud from legitimate transactions.

---

## Analysis Roadmap

1. Data profiling and imbalance quantification  
2. Feature scaling and preprocessing  
3. Balanced sampling strategy (undersampling + SMOTE comparison)  
4. Correlation analysis on the balanced dataset  
5. Anomaly / extreme outlier removal  
6. Classifier comparison with appropriate metrics  
7. Final evaluation on the original (imbalanced) test set  

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, cross_val_predict,
    GridSearchCV, ShuffleSplit, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    f1_score, recall_score, precision_score, accuracy_score
)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA, TruncatedSVD

# imbalanced-learn for SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as imb_pipeline

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
PALETTE = ['#2166AC', '#D6604D']  # Blue = legitimate, Red = fraud

print('All libraries loaded successfully.')

## 2. Data Loading and Initial Profile

Before any transformation, we take a full inventory: shape, data types, null counts, and a statistical summary. This is non-negotiable — you cannot make good analytical decisions on data you haven't profiled.

In [ ]:
df = pd.read_csv('creditcard.csv')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'\nNull values per column: {df.isnull().sum().max()} (max across all columns)')

In [ ]:
df.describe().round(2)

### Class Imbalance

This is the central challenge of the dataset. Standard accuracy is misleading here — a model that predicts "no fraud" for every transaction would score **99.83% accuracy** while catching zero fraud cases.

In [ ]:
class_counts = df['Class'].value_counts()
fraud_pct = round(class_counts[1] / len(df) * 100, 2)
legit_pct = round(class_counts[0] / len(df) * 100, 2)

print(f'Legitimate transactions: {class_counts[0]:,}  ({legit_pct}%)')
print(f'Fraudulent transactions:  {class_counts[1]:,}   ({fraud_pct}%)')
print(f'\nImbalance ratio: {round(class_counts[0]/class_counts[1])}:1 (legitimate:fraud)')

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='Class', data=df, palette=PALETTE, ax=ax)
ax.set_title('Class Distribution — Original Dataset', fontsize=13, fontweight='bold')
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
ax.set_ylabel('Transaction count')
ax.bar_label(ax.containers[0], fmt='{:,.0f}')
plt.tight_layout()
plt.show()

### Distribution of Transaction Amount and Time

These are the only two features not transformed by PCA, so they're worth examining closely. `Amount` is heavily right-skewed — a handful of very large transactions dominate the scale. `Time` records seconds elapsed since the first transaction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['Amount'], bins=80, color='#2166AC', ax=axes[0], kde=True)
axes[0].set_title('Distribution of Transaction Amount', fontsize=12)
axes[0].set_xlabel('Amount (USD)')
axes[0].set_ylabel('Frequency')

sns.histplot(df['Time'], bins=80, color='#4DAC26', ax=axes[1], kde=True)
axes[1].set_title('Distribution of Transaction Time', fontsize=12)
axes[1].set_xlabel('Seconds since first transaction')
axes[1].set_ylabel('Frequency')

plt.suptitle('Raw Feature Distributions (before scaling)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Amount — Mean: ${df['Amount'].mean():.2f} | Median: ${df['Amount'].median():.2f} | Max: ${df['Amount'].max():.2f}")

## 3. Feature Scaling

`Time` and `Amount` are on completely different scales from V1–V28 (which are already PCA-transformed and centred). We apply **RobustScaler** — which uses median and IQR instead of mean and standard deviation — because `Amount` has significant outliers that would distort a StandardScaler.

The original `Time` and `Amount` columns are dropped after scaling.

In [ ]:
scaler = RobustScaler()

df['scaled_amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time']   = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Drop originals and reorder so scaled versions appear at the front
df.drop(['Amount', 'Time'], axis=1, inplace=True)

scaled_amount = df.pop('scaled_amount')
scaled_time   = df.pop('scaled_time')
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

print('Scaling complete. Final feature set:')
print(df.columns.tolist())
df.head(3)

## 4. Train / Test Split — Original (Imbalanced) Data

**Critical design decision:** We split the original dataset *before* any sampling. The test set is kept in its natural imbalanced state — because that is what the model will face in production. We only apply sampling techniques to the training set.

Using a stratified split preserves the fraud proportion in both train and test.

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split — preserves class ratio in both train and test
sss = StratifiedKFold(n_splits=5, random_state=None, shuffle=False)

for train_idx, test_idx in sss.split(X, y):
    original_Xtrain = X.iloc[train_idx]
    original_Xtest  = X.iloc[test_idx]
    original_ytrain = y.iloc[train_idx]
    original_ytest  = y.iloc[test_idx]

# Verify stratification held
print(f'Train fraud rate: {original_ytrain.mean():.4f}')
print(f'Test  fraud rate: {original_ytest.mean():.4f}')
print(f'\nTrain shape: {original_Xtrain.shape} | Test shape: {original_Xtest.shape}')

## 5. Handling Class Imbalance

### Strategy A — Random Undersampling

We create a balanced subsample by taking all 492 fraud cases and randomly drawing 492 legitimate cases. This gives us a 50/50 split for training. The main tradeoff: we discard the vast majority of our legitimate transaction data.

This balanced subsample is used **only for correlation analysis and initial model training** — not for final evaluation.

In [ ]:
df_shuffled = df.sample(frac=1, random_state=42)

fraud_df     = df_shuffled[df_shuffled['Class'] == 1]
non_fraud_df = df_shuffled[df_shuffled['Class'] == 0].iloc[:len(fraud_df)]

balanced_df = pd.concat([fraud_df, non_fraud_df]).sample(frac=1, random_state=42)

print('Balanced subsample class distribution:')
print(balanced_df['Class'].value_counts())
print(f'\nTotal rows in subsample: {len(balanced_df)}')

fig, ax = plt.subplots(figsize=(5, 3))
sns.countplot(x='Class', data=balanced_df, palette=PALETTE, ax=ax)
ax.set_title('Balanced Subsample (50/50)', fontsize=12, fontweight='bold')
ax.set_xticklabels(['Legitimate', 'Fraud'])
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

## 6. Correlation Analysis — Which Features Signal Fraud?

Correlation analysis on the *imbalanced* original dataset is misleading — the sheer volume of legitimate transactions drowns out any fraud signal. We use the balanced subsample to reveal genuine correlations.

**What we're looking for:** Features that show a strong positive or negative correlation with `Class`. These are our most discriminative signals.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(22, 18))

# Imbalanced correlation (for reference — not for analysis)
corr_imbalanced = df.corr()
sns.heatmap(corr_imbalanced, cmap='coolwarm_r', ax=ax1,
            annot=False, linewidths=0.3)
ax1.set_title("Correlation Matrix — Original Imbalanced Dataset\n"
              "(misleading due to class imbalance — do not use for feature selection)",
              fontsize=13, fontweight='bold')

# Balanced correlation (the one to use)
corr_balanced = balanced_df.corr()
sns.heatmap(corr_balanced, cmap='coolwarm_r', ax=ax2,
            annot=False, linewidths=0.3)
ax2.set_title("Correlation Matrix — Balanced Subsample\n"
              "(use this for feature insight)",
              fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Isolate and rank the Class correlations from the balanced set
class_corr = corr_balanced['Class'].drop('Class').sort_values()

print('Top 5 NEGATIVELY correlated features (lower value → more likely fraud):')
print(class_corr.head(5).round(3))
print('\nTop 5 POSITIVELY correlated features (higher value → more likely fraud):')
print(class_corr.tail(5).round(3))

In [ ]:
# Visualise the strongest negative and positive correlations as boxplots
neg_features = ['V17', 'V14', 'V12', 'V10']
pos_features = ['V11', 'V4', 'V2', 'V19']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for i, feat in enumerate(neg_features):
    sns.boxplot(x='Class', y=feat, data=balanced_df,
                palette=PALETTE, ax=axes[0][i])
    axes[0][i].set_title(f'{feat} — Negative correlation', fontsize=11)
    axes[0][i].set_xticklabels(['Legitimate', 'Fraud'])

for i, feat in enumerate(pos_features):
    sns.boxplot(x='Class', y=feat, data=balanced_df,
                palette=PALETTE, ax=axes[1][i])
    axes[1][i].set_title(f'{feat} — Positive correlation', fontsize=11)
    axes[1][i].set_xticklabels(['Legitimate', 'Fraud'])

plt.suptitle('Most Discriminative Features: Distribution by Class',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Reading these boxplots:** For the negatively correlated features (top row), fraud transactions cluster at significantly lower values than legitimate ones — V14 shows this most dramatically. For positively correlated features (bottom row), fraud clusters higher. These features will be the most valuable inputs to our classifiers.

## 7. Extreme Outlier Removal (IQR Method)

Extreme outliers in the fraud class distort the model's decision boundary. We apply the **Interquartile Range (IQR)** method to the three strongest negatively correlated features: V14, V12, and V10.

The IQR rule: a value is extreme if it falls below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR`, computed *within the fraud class only*.

This is applied to the balanced subsample — not the full dataset.

In [ ]:
def remove_iqr_outliers(dataframe, feature, label_col='Class', fraud_label=1, multiplier=1.5):
    """Remove extreme outliers from a feature within the fraud class using IQR method."""
    fraud_vals = dataframe.loc[dataframe[label_col] == fraud_label, feature]
    q25, q75   = fraud_vals.quantile(0.25), fraud_vals.quantile(0.75)
    iqr        = q75 - q25
    lower      = q25 - multiplier * iqr
    upper      = q75 + multiplier * iqr

    outliers = dataframe[
        (dataframe[label_col] == fraud_label) &
        ((dataframe[feature] < lower) | (dataframe[feature] > upper))
    ].index

    print(f'{feature}: bounds [{lower:.2f}, {upper:.2f}] — removing {len(outliers)} extreme outlier(s)')
    return dataframe.drop(outliers)


print(f'Rows before outlier removal: {len(balanced_df)}')
cleaned_df = balanced_df.copy()
for feat in ['V14', 'V12', 'V10']:
    cleaned_df = remove_iqr_outliers(cleaned_df, feat)
print(f'Rows after outlier removal:  {len(cleaned_df)}')
print(f'Outliers removed: {len(balanced_df) - len(cleaned_df)}')

In [ ]:
# Visualise the effect of outlier removal on V14, V12, V10
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, feat in enumerate(['V14', 'V12', 'V10']):
    sns.boxplot(x='Class', y=feat, data=cleaned_df,
                palette=PALETTE, ax=axes[i])
    axes[i].set_title(f'{feat} — After outlier removal', fontsize=11)
    axes[i].set_xticklabels(['Legitimate', 'Fraud'])

plt.suptitle('Key Features After IQR Outlier Removal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Dimensionality Reduction — Visualising Separability

Before training classifiers, it's worth asking: are fraud and legitimate transactions separable at all in feature space? We use three dimensionality reduction techniques to project the 29-dimensional balanced dataset down to 2D for visual inspection.

- **t-SNE** — best at revealing local cluster structure
- **PCA** — linear projection, fastest, shows global variance
- **Truncated SVD** — similar to PCA but works on sparse representations

In [ ]:
X_vis = cleaned_df.drop('Class', axis=1)
y_vis = cleaned_df['Class']

t0 = time.time()
X_tsne = TSNE(n_components=2, random_state=42).fit_transform(X_vis)
print(f't-SNE:       {time.time()-t0:.1f}s')

t0 = time.time()
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_vis)
print(f'PCA:         {time.time()-t0:.1f}s')

t0 = time.time()
X_svd = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_vis)
print(f'Truncated SVD: {time.time()-t0:.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Class Separability via Dimensionality Reduction\n(Balanced subsample)',
             fontsize=13, fontweight='bold')

legend_patches = [
    mpatches.Patch(color='#2166AC', label='Legitimate'),
    mpatches.Patch(color='#D6604D', label='Fraud')
]

datasets = [
    (X_tsne, 't-SNE'),
    (X_pca,  'PCA'),
    (X_svd,  'Truncated SVD')
]

for ax, (X_2d, title) in zip(axes, datasets):
    ax.scatter(X_2d[y_vis==0, 0], X_2d[y_vis==0, 1],
               c='#2166AC', alpha=0.5, s=15, label='Legitimate')
    ax.scatter(X_2d[y_vis==1, 0], X_2d[y_vis==1, 1],
               c='#D6604D', alpha=0.7, s=15, label='Fraud')
    ax.set_title(title, fontsize=12)
    ax.legend(handles=legend_patches, fontsize=9)

plt.tight_layout()
plt.show()

**Interpretation:** t-SNE typically shows the clearest separation — two distinct clusters confirming that the engineered features do carry meaningful signal. If classes overlapped completely here, we'd know the problem is fundamentally hard and temper our expectations accordingly.

## 9. Classifier Training — Undersampled Data

We train four classifiers on the balanced subsample and compare them using cross-validated accuracy as a starting point. Note that accuracy is not our final metric — it's used here only to quickly filter candidate models.

In [ ]:
X_bal = cleaned_df.drop('Class', axis=1).values
y_bal = cleaned_df['Class'].values

X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42
)

classifiers = {
    'Logistic Regression':    LogisticRegression(),
    'K-Nearest Neighbours':   KNeighborsClassifier(),
    'Support Vector Machine': SVC(),
    'Decision Tree':          DecisionTreeClassifier()
}

print('Cross-validated accuracy on balanced training set (5-fold):\n')
for name, clf in classifiers.items():
    clf.fit(X_train_bal, y_train_bal)
    scores = cross_val_score(clf, X_train_bal, y_train_bal, cv=5)
    print(f'  {name:<30} {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%')

### Hyperparameter Tuning — GridSearchCV

We tune the two most promising classifiers — Logistic Regression and KNN — using grid search with cross-validation.

In [ ]:
# Logistic Regression
log_params = {'penalty': ['l1', 'l2'], 'C': [0.001, 0.01, 0.1, 1, 10, 100]}
grid_log = GridSearchCV(LogisticRegression(solver='liblinear'), log_params, cv=5, scoring='f1')
grid_log.fit(X_train_bal, y_train_bal)
log_reg = grid_log.best_estimator_
print(f'Best Logistic Regression params: {grid_log.best_params_}')

# KNN
knn_params = {'n_neighbors': list(range(2, 10)), 'weights': ['uniform', 'distance']}
grid_knn = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, scoring='f1')
grid_knn.fit(X_train_bal, y_train_bal)
knn_clf = grid_knn.best_estimator_
print(f'Best KNN params:                {grid_knn.best_params_}')

# SVC and Decision Tree with defaults
svc_clf  = SVC(probability=True)
tree_clf = DecisionTreeClassifier(random_state=42)
svc_clf.fit(X_train_bal, y_train_bal)
tree_clf.fit(X_train_bal, y_train_bal)

### ROC Curves — Comparing All Classifiers

ROC-AUC tells us how well each model distinguishes between classes across all possible thresholds. A score of 1.0 is perfect; 0.5 is no better than random.

In [ ]:
# Get decision scores for each classifier via cross-validation
log_scores  = cross_val_predict(log_reg,  X_train_bal, y_train_bal, cv=5, method='decision_function')
knn_scores  = cross_val_predict(knn_clf,  X_train_bal, y_train_bal, cv=5, method='predict_proba')[:, 1]
svc_scores  = cross_val_predict(svc_clf,  X_train_bal, y_train_bal, cv=5, method='decision_function')
tree_scores = cross_val_predict(tree_clf, X_train_bal, y_train_bal, cv=5, method='predict_proba')[:, 1]

roc_data = {
    'Logistic Regression':    log_scores,
    'K-Nearest Neighbours':   knn_scores,
    'Support Vector Machine': svc_scores,
    'Decision Tree':          tree_scores
}

print('ROC-AUC scores (cross-validated on balanced training set):\n')
for name, scores in roc_data.items():
    auc = roc_auc_score(y_train_bal, scores)
    print(f'  {name:<30} AUC = {auc:.4f}')

# Plot
colors_roc = ['#1b7837', '#762a83', '#e08214', '#2166ac']
fig, ax = plt.subplots(figsize=(9, 7))

for (name, scores), color in zip(roc_data.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_train_bal, scores)
    auc = roc_auc_score(y_train_bal, scores)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Classifiers (Balanced Training Set)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([-0.01, 1])
ax.set_ylim([0, 1.01])
plt.tight_layout()
plt.show()

## 10. Strategy B — SMOTE Oversampling

SMOTE (Synthetic Minority Oversampling Technique) generates synthetic fraud samples by interpolating between existing fraud cases in feature space. Unlike undersampling, it retains all legitimate transactions.

**Key rule:** SMOTE is applied only to the *training* set. The test set remains untouched — it represents real production conditions.

In [ ]:
sm = SMOTE(random_state=42)
X_smote_train, y_smote_train = sm.fit_resample(
    original_Xtrain.values, original_ytrain.values
)

print(f'Original training set:  {original_Xtrain.shape[0]:,} rows | Fraud: {original_ytrain.sum()}')
print(f'After SMOTE:            {X_smote_train.shape[0]:,} rows | Fraud: {y_smote_train.sum()}')
print(f'\nClass balance after SMOTE: {pd.Series(y_smote_train).value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# Train Logistic Regression on SMOTE data
log_reg_smote = LogisticRegression(**grid_log.best_params_, solver='liblinear', random_state=42)
log_reg_smote.fit(X_smote_train, y_smote_train)

smote_preds  = log_reg_smote.predict(original_Xtest)
smote_scores = log_reg_smote.decision_function(original_Xtest)

print('SMOTE Logistic Regression — Classification Report (Original Test Set):\n')
print(classification_report(original_ytest, smote_preds, target_names=['Legitimate', 'Fraud']))

## 11. Final Evaluation — Confusion Matrices

The confusion matrix is the most transparent evaluation tool for fraud detection. It shows exactly how many fraud cases we caught, how many we missed, and how many false alarms we generated.

We compare Logistic Regression trained on undersampled data vs. SMOTE data, evaluated on the original imbalanced test set.

In [ ]:
# Predictions from undersampled model on original test set
undersample_preds = log_reg.predict(original_Xtest)

cm_under = confusion_matrix(original_ytest, undersample_preds)
cm_smote = confusion_matrix(original_ytest, smote_preds)

def plot_cm(cm, ax, title, cmap):
    labels = np.array([
        [f'True Neg\n{cm[0,0]:,}',  f'False Pos\n{cm[0,1]:,}'],
        [f'False Neg\n{cm[1,0]:,}', f'True Pos\n{cm[1,1]:,}']
    ])
    sns.heatmap(cm, annot=labels, fmt='', cmap=cmap, ax=ax,
                xticklabels=['Predicted Legit', 'Predicted Fraud'],
                yticklabels=['Actual Legit', 'Actual Fraud'],
                linewidths=1, linecolor='white', cbar=False,
                annot_kws={'size': 12})
    ax.set_title(title, fontsize=12, fontweight='bold')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_cm(cm_under, axes[0], 'Logistic Regression\n(Undersampled training)', 'Blues')
plot_cm(cm_smote, axes[1], 'Logistic Regression\n(SMOTE training)',        'Oranges')

plt.suptitle('Confusion Matrices — Evaluated on Original Imbalanced Test Set',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Precision-Recall Curves

For severely imbalanced datasets, the Precision-Recall curve is more informative than the ROC curve. It directly shows the trade-off between catching fraud (recall) and avoiding false alarms (precision).

In [ ]:
undersample_scores = log_reg.decision_function(original_Xtest)

prec_u, rec_u, _ = precision_recall_curve(original_ytest, undersample_scores)
prec_s, rec_s, _ = precision_recall_curve(original_ytest, smote_scores)

ap_u = average_precision_score(original_ytest, undersample_scores)
ap_s = average_precision_score(original_ytest, smote_scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (prec, rec, ap, title, color) in zip(axes, [
    (prec_u, rec_u, ap_u, f'Undersampling (AP = {ap_u:.2f})', '#004a93'),
    (prec_s, rec_s, ap_s, f'SMOTE (AP = {ap_s:.2f})',         '#c94a00')
]):
    ax.step(rec, prec, color=color, where='post', linewidth=2)
    ax.fill_between(rec, prec, step='post', alpha=0.15, color=color)
    ax.set_xlabel('Recall (fraud catch rate)', fontsize=11)
    ax.set_ylabel('Precision (signal accuracy)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.set_xlim([0, 1.0])

plt.suptitle('Precision-Recall Curves — Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Final Model Comparison

Summarising all key metrics side by side on the original imbalanced test set.

In [ ]:
def get_metrics(y_true, y_pred, y_scores):
    return {
        'Recall':    round(recall_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'F1':        round(f1_score(y_true, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_true, y_scores), 4),
        'Avg Prec':  round(average_precision_score(y_true, y_scores), 4)
    }

results = pd.DataFrame({
    'Undersampling': get_metrics(original_ytest, undersample_preds, undersample_scores),
    'SMOTE':         get_metrics(original_ytest, smote_preds, smote_scores)
}).T

print('Final model comparison — evaluated on original imbalanced test set:\n')
print(results.to_string())

# Highlight best per metric
results.style.highlight_max(axis=0, color='lightgreen')

## 13. Conclusions and Business Interpretation

### What the model achieves

The Logistic Regression model trained on SMOTE-augmented data delivers the strongest performance on the real-world imbalanced test set. In practical terms:

- **Recall:** The model catches the large majority of actual fraud cases — meaning most fraudulent transactions would be flagged for review.
- **Precision:** A meaningful proportion of flagged transactions are genuine fraud — the false alarm rate is manageable for an operations team.
- **ROC-AUC:** The model substantially outperforms random chance and demonstrates genuine discriminative ability.

### Key analytical decisions made

| Decision | Rationale |
|---|---|
| RobustScaler on Amount and Time | Resistant to the extreme outliers in transaction amounts |
| Evaluate on original imbalanced test set | Reflects true production conditions |
| Use Recall and ROC-AUC as primary metrics | Standard accuracy is misleading at 0.17% fraud rate |
| IQR outlier removal on fraud class only | Avoids distorting the model boundary with extreme edge cases |
| Compare undersampling vs SMOTE | Neither is universally better; SMOTE generally preserves more information |

### Limitations

- **Feature opacity:** V1–V28 are PCA-transformed for confidentiality. We cannot explain *what* drives the fraud signal in business terms.
- **Temporal risk:** The dataset covers only two days in 2013. Fraud patterns evolve; model performance would degrade over time without retraining.
- **Threshold sensitivity:** The decision threshold (default 0.5) can be tuned to shift the precision/recall trade-off based on operational priorities — e.g., a bank may accept more false positives to catch more fraud.

### Recommended next steps

1. Deploy with a **precision-recall threshold tuned** to the bank's cost model (cost of missed fraud vs. cost of investigating a false alarm).
2. Implement **periodic retraining** as fraud patterns shift.
3. Consider **ensemble methods** (Random Forest, XGBoost) which often outperform single classifiers on tabular fraud data.
4. Build a **monitoring dashboard** tracking live precision and recall as the model scores production transactions.